In [ ]:
%run _bootstrap_dev.ipynb

# Lazy Portfolio Analyst
Flusso: frequency selection → frontiera efficiente → backtest PTF
proposto → backtest PTF ottimizzato.
Per la validazione statistica (MC, Stability, DSR) vedere §6 (da implementare).

## §1 — Configurazione

In [ ]:
# Portafoglio da analizzare (definiti in l_portfolios.py)
my_portfolio       = greta_base_spy_portfolio_etf_ita
my_portfolio_title = "Gretchen S&P 500 Core-Engine - Fiduciaria"

# Parametri analisi
start_date = '2016-01-01'   # inizio backtest
end_date   = None           # None = oggi
benchmark  = 'SPY'
init_cash  = 100_000
fees       = 0.001
years      = 10             # finestra frontiera efficiente (anni)

## §2 — Frequency Selection
Testa W/M/Q/Y/None e seleziona la frequenza con Sharpe massimo.
Cambia `freq_selection_metric` per usare 'cagr' | 'total_return' | 'max_dd'.

In [ ]:
freqs = ['W', 'M', 'Q', 'Y', None]
_rows = []
for _freq in freqs:
    _pf = run_bh_backtest(my_portfolio, start_date, end_date,
                          init_cash, fees, _freq)
    _eq = _pf.value()
    if isinstance(_eq, pd.DataFrame): _eq = _eq.iloc[:,0]
    _eq = _eq.dropna()
    _yrs = len(_eq) / 252
    _cagr = (_eq.iloc[-1]/_eq.iloc[0])**(1/_yrs)-1 if _yrs > 0 else np.nan
    _rows.append({
        'Freq'        : _freq if _freq is not None else 'BH',
        'Sharpe'      : float(_pf.sharpe_ratio()),
        'CAGR%'       : round(_cagr * 100, 2),
        'TotalReturn%': round(float(_pf.total_return()) * 100, 2),
        'MaxDD%'      : round(abs(float(_pf.max_drawdown())) * 100, 2),
    })
freq_df = pd.DataFrame(_rows)
my_display(freq_df, title="Confronto frequenze di ribilanciamento")

# Selezione automatica: max Sharpe (modifica idxmax → idxmin per MaxDD%)
freq_selection_metric = 'Sharpe'
best_label = freq_df.loc[freq_df[freq_selection_metric].idxmax(), 'Freq']
best_freq  = None if best_label == 'BH' else best_label
print(f"\n✅ Frequenza ottimale ({freq_selection_metric}): {best_label}")

## §3 — Frontiera efficiente
Posiziona il PTF proposto rispetto alla frontiera efficiente e
suggerisce i pesi ottimali (Max Sharpe).

**Legenda colonne:**
- `Return` / `Volatility` / `Sharpe` — valori teorici stimati da
  media storica e covarianza (PyPortfolioOpt, in-sample)
- `Real Return` / `Real Volatility` / `Real Sharpe` — valori
  effettivi misurati sul backtest reale nel periodo selezionato
  (out-of-sample rispetto all'ottimizzazione)

In [ ]:
my_tickers = list(my_portfolio.keys())
my_weights = list(my_portfolio.values())

fig_frontier, df_special = efficient_frontier_pypfopt(
    tickers=my_tickers,
    years=years,
    my_weights=my_weights,
    n_points=80,
    weight_bounds=(0, 1),
    show_plot=True,
    interactive=True,
    print_weights=True,
)
fig_frontier.show()

# Pesi ottimali Max Sharpe
_metric_cols = {'Return', 'Volatility', 'Sharpe',
                'Real Return', 'Real Volatility', 'Real Sharpe'}
_weight_cols = [c for c in df_special.columns if c not in _metric_cols]
optimal_weights = df_special.loc['Max Sharpe', _weight_cols].to_dict()
print("\nPesi ottimali (Max Sharpe):")
for t, w in optimal_weights.items():
    print(f"  {t:15s}: {w:.2%}")

## §4 — Backtest PTF proposto
Analisi completa con la frequenza ottimale selezionata in §2.

In [ ]:
benchmark_data = download_data(benchmark, start_date, end_date)

pf_proposed = run_bh_backtest(my_portfolio, start_date, end_date,
                               init_cash, fees, best_freq)
_ = generate_lazy_portfolio_performance(
    pf=pf_proposed,
    portfolio_title=my_portfolio_title,
    benchmark=benchmark,
    benchmark_data=benchmark_data,
    show_report=True,
    show_plots=True,
    alpha_analysis=True,
)

## §5 — Backtest PTF ottimizzato
Stesso backtest con i pesi ottimali suggeriti dalla frontiera in §3.
Confronta con §4 per valutare il guadagno dell'ottimizzazione.

In [ ]:
pf_optimal = run_bh_backtest(optimal_weights, start_date, end_date,
                              init_cash, fees, best_freq)
_ = generate_lazy_portfolio_performance(
    pf=pf_optimal,
    portfolio_title=my_portfolio_title + ' — Ottimizzato (Max Sharpe)',
    benchmark=benchmark,
    benchmark_data=benchmark_data,
    show_report=True,
    show_plots=True,
    alpha_analysis=True,
)

## §6 — Validazione statistica
*(Da implementare — Fase B)*
- Stability test pesi ottimali su sotto-periodi
- Monte Carlo Block A: confidence intervals su CAGR/Sharpe/DD
- Monte Carlo Block B: skill del ribilanciamento vs BH puro
- DSR: Deflated Sharpe Ratio
- Decisione finale: promuovi/rigetta

In [ ]:
# Placeholder — da implementare
pass